In [ ]:
import os
import random
import time
from functools import partial
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import scipy.io as scio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from datasets.arrow_dataset import Dataset as dsds
from mel_cepstral_distance import get_metrics_mels
from sklearn.model_selection import KFold
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torchinfo import summary
from tqdm.notebook import tqdm

SEED = 773
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device, 'torch_version=' + torch.__version__)


In [ ]:
def window_sumsquare(window, n_frames, hop_length=200, win_length=800,
                     n_fft=800, dtype=np.float32, norm=None):
    """
    # from librosa 0.6
    Compute the sum-square envelope of a window function at a given hop length.

    This is used to estimate modulation effects induced by windowing
    observations in short-time fourier transforms.

    Parameters
    ----------
    window : string, tuple, number, callable, or list-like
        Window specification, as in `get_window`

    n_frames : int > 0
        The number of analysis frames

    hop_length : int > 0
        The number of samples to advance between frames

    win_length : [optional]
        The length of the window function.  By default, this matches `n_fft`.

    n_fft : int > 0
        The length of each analysis frame.

    dtype : np.dtype
        The data type of the output

    Returns
    
    -------
    wss : np.ndarray, shape=`(n_fft + hop_length * (n_frames - 1))`
        The sum-squared envelope of the window function
    """
    if win_length is None:
        win_length = n_fft

    n = n_fft + hop_length * (n_frames - 1)
    x = np.zeros(n, dtype=dtype)

    # Compute the squared window at the requested length.
    win_sq = get_window(window, win_length, fftbins=True)
    win_sq = librosa.util.normalize(win_sq, norm=norm)**2
    win_sq = librosa.util.pad_center(win_sq, size=n_fft)

    # Accumulate the window-sum-square envelope.
    for i in range(n_frames):
        sample = i * hop_length
        x[sample:min(n, sample + n_fft)] += win_sq[:max(0, min(n_fft, n - sample))]
    return x


def griffin_lim(magnitudes, stft_fn, n_iters=30):
    """
    PARAMS
    ------
    magnitudes: spectrogram magnitudes
    stft_fn: STFT class with transform (STFT) and inverse (ISTFT) methods
    """

    angles = np.angle(np.exp(2j * np.pi * np.random.rand(*magnitudes.size())))
    angles = angles.astype(np.float32)
    angles = torch.autograd.Variable(torch.from_numpy(angles))
    signal = stft_fn.inverse(magnitudes, angles).squeeze(1)

    for i in range(n_iters):
        _, angles = stft_fn.transform(signal)
        signal = stft_fn.inverse(magnitudes, angles).squeeze(1)
    return signal


def dynamic_range_compression(x, C=1, clip_val=1e-5):
    """
    PARAMS
    ------
    C: compression factor
    """
    return torch.log(torch.clamp(x, min=clip_val) * C)


def dynamic_range_decompression(x, C=1):
    """
    PARAMS
    ------
    C: compression factor used to compress
    """
    return torch.exp(x) / C

In [ ]:
import torch
import numpy as np
import torch.nn.functional as F
from torch.autograd import Variable
from scipy.signal import get_window
from librosa.util import pad_center, tiny
class STFT(torch.nn.Module):
    """adapted from Prem Seetharaman's https://github.com/pseeth/pytorch-stft"""
    def __init__(self, filter_length=800, hop_length=200, win_length=800,
                 window='hann'):
        super(STFT, self).__init__()
        self.filter_length = filter_length
        self.hop_length = hop_length
        self.win_length = win_length
        self.window = window
        self.forward_transform = None
        scale = self.filter_length / self.hop_length
        fourier_basis = np.fft.fft(np.eye(self.filter_length))

        cutoff = int((self.filter_length / 2 + 1))
        fourier_basis = np.vstack([np.real(fourier_basis[:cutoff, :]),
                                   np.imag(fourier_basis[:cutoff, :])])

        forward_basis = torch.FloatTensor(fourier_basis[:, None, :])
        inverse_basis = torch.FloatTensor(
            np.linalg.pinv(scale * fourier_basis).T[:, None, :])

        if window is not None:
            assert(filter_length >= win_length)
            # Obtain the window and zero-pad it symmetrically to filter_length.
            fft_window = get_window(window, win_length, fftbins=True)
            fft_window = pad_center(data=fft_window, size=filter_length)
            fft_window = torch.from_numpy(fft_window).float()

            # Apply the window to each Fourier basis.
            forward_basis *= fft_window
            inverse_basis *= fft_window

        self.register_buffer('forward_basis', forward_basis.float())
        self.register_buffer('inverse_basis', inverse_basis.float())

    def transform(self, input_data):
        num_batches = input_data.size(0)
        num_samples = input_data.size(1)

        self.num_samples = num_samples

        # Reflect-pad the input to match librosa's behavior.
        input_data = input_data.view(num_batches, 1, num_samples)
        input_data = F.pad(
            input_data.unsqueeze(1),
            (int(self.filter_length / 2), int(self.filter_length / 2), 0, 0),
            mode='reflect')
        input_data = input_data.squeeze(1)

        # https://github.com/NVIDIA/tacotron2/issues/125
        forward_transform = F.conv1d(
            input_data.cuda(),
            Variable(self.forward_basis, requires_grad=False).cuda(),
            stride=self.hop_length,
            padding=0).cpu()

        cutoff = int((self.filter_length / 2) + 1)
        real_part = forward_transform[:, :cutoff, :]
        imag_part = forward_transform[:, cutoff:, :]

        magnitude = torch.sqrt(real_part**2 + imag_part**2)
        phase = torch.autograd.Variable(
            torch.atan2(imag_part.data, real_part.data))

        return magnitude, phase

    def inverse(self, magnitude, phase):
        recombine_magnitude_phase = torch.cat(
            [magnitude*torch.cos(phase), magnitude*torch.sin(phase)], dim=1)

        inverse_transform = F.conv_transpose1d(
            recombine_magnitude_phase,
            Variable(self.inverse_basis, requires_grad=False),
            stride=self.hop_length,
            padding=0)

        if self.window is not None:
            window_sum = window_sumsquare(
                self.window, magnitude.size(-1), hop_length=self.hop_length,
                win_length=self.win_length, n_fft=self.filter_length,
                dtype=np.float32)
            # Remove modulation effects.
            approx_nonzero_indices = torch.from_numpy(
                np.where(window_sum > tiny(window_sum))[0])
            window_sum = torch.autograd.Variable(
                torch.from_numpy(window_sum), requires_grad=False)
            window_sum = window_sum.cuda() if magnitude.is_cuda else window_sum
            inverse_transform[:, :, approx_nonzero_indices] /= window_sum[approx_nonzero_indices]

            # Scale by the hop-length ratio.
            inverse_transform *= float(self.filter_length) / self.hop_length

        inverse_transform = inverse_transform[:, :, int(self.filter_length/2):]
        inverse_transform = inverse_transform[:, :, :-int(self.filter_length/2):]

        return inverse_transform

    def forward(self, input_data):
        self.magnitude, self.phase = self.transform(input_data)
        reconstruction = self.inverse(self.magnitude, self.phase)
        return reconstruction


class TacotronSTFT(torch.nn.Module):
    def __init__(self, filter_length=1024, hop_length=256, win_length=1024,
                 n_mel_channels=80, sampling_rate=22050, mel_fmin=0.0,
                 mel_fmax=None):
        super(TacotronSTFT, self).__init__()
        self.n_mel_channels = n_mel_channels
        self.sampling_rate = sampling_rate
        self.stft_fn = STFT(filter_length, hop_length, win_length)
        mel_basis = librosa.filters.mel(
            sr=sampling_rate,
            n_fft=filter_length,
            n_mels=n_mel_channels,
            fmin=mel_fmin,
            fmax=mel_fmax
        )

        mel_basis = torch.from_numpy(mel_basis).float()
        self.register_buffer('mel_basis', mel_basis)

    def spectral_normalize(self, magnitudes):
        output = dynamic_range_compression(magnitudes)
        return output

    def spectral_de_normalize(self, magnitudes):
        output = dynamic_range_decompression(magnitudes)
        return output

    def mel_spectrogram(self, y):
        """Computes mel-spectrograms from a batch of waves
        PARAMS
        ------
        y: Variable(torch.FloatTensor) with shape (B, T) in range [-1, 1]

        RETURNS
        -------
        mel_output: torch.FloatTensor of shape (B, n_mel_channels, T)
        """
        assert(torch.min(y.data) >= -1)
        assert(torch.max(y.data) <= 1)

        magnitudes, phases = self.stft_fn.transform(y)
        magnitudes = magnitudes.data
        mel_output = torch.matmul(self.mel_basis, magnitudes)
        mel_output = self.spectral_normalize(mel_output)
        return mel_output

In [ ]:
n_mel_channels=80
segment_length=33075
pad_short=2000
filter_length=1024
hop_length=256 # This value must remain unchanged for checkpoint compatibility.
win_length=1024
sampling_rate=22050
mel_fmin=0.0
mel_fmax=8000.0
stft = TacotronSTFT(filter_length=filter_length,
                        hop_length=hop_length,
                        win_length=win_length,
                        n_mel_channels=n_mel_channels,
                        sampling_rate=sampling_rate,
                        mel_fmin=mel_fmin,
                        mel_fmax=mel_fmax)

In [ ]:

# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# class BasicBlock(nn.Module):
#     def __init__(self, in_channels, out_channels, stride=1):
#         super(BasicBlock, self).__init__()
#         self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
#         self.bn1 = nn.BatchNorm2d(out_channels)
#         self.relu = nn.ReLU(inplace=True)
#         self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
#         self.bn2 = nn.BatchNorm2d(out_channels)
        
#         self.downsample = nn.Sequential()
#         if stride != 1 or in_channels != out_channels:
#             self.downsample = nn.Sequential(
#                 nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
#                 nn.BatchNorm2d(out_channels)
#             )

#     def forward(self, x):
#         identity = self.downsample(x)
#         out = self.conv1(x)
#         out = self.bn1(out)
#         out = self.relu(out)
#         out = self.conv2(out)
#         out = self.bn2(out)
#         out += identity
#         out = self.relu(out)
#         return out

# class CustomResNet(nn.Module):
#     def __init__(self):
#         super(CustomResNet, self).__init__()
#         self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=1, padding=1)
        
#         self.layer1 = BasicBlock(64, 64)
        
#         self.pool0 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)


#         self.conv2 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)

#         self.layer2 = BasicBlock(64, 64)

#         self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0) 

#         self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)

#         self.layer3 = BasicBlock(64, 64)

#         self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
#         self.conv4 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)

#         self.layer4 = BasicBlock(64, 64)

#         self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
#         #self.pool3 = nn.AdaptiveAvgPool2d((5, 12))  # replaces fixed-size pooling

#         self.fc = nn.Linear(64 * 5 * 12, 80 * 130)

#     def forward(self, x):
#         x = self.conv1(x)  
#         x = self.layer1(x)
#         x = self.pool0(x)
#         x = self.conv2(x)

#         x = self.layer2(x)
#         x = self.pool1(x)
#         x = self.conv3(x)
#         x = self.layer3(x)

        
#         x = self.pool2(x)
#         x = self.conv4(x)

#         x = self.layer4(x)
#         x = self.pool3(x)
#         x = x.view(x.size(0), -1)
#         x = self.fc(x)
#         x = x.view(-1, 80, 130)
#         return x

# # Example usage:
# model = CustomResNet()
# input_tensor = torch.randn(1, 1, 94, 200)
# output_tensor = model(input_tensor)
# print(output_tensor.shape)



import torch
import torch.nn as nn
import torch.nn.functional as F

# BasicBlock is unchanged.
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.downsample = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.downsample(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += identity
        out = self.relu(out)
        return out

# CustomResNet with an optional fc_in_features argument.
class CustomResNet(nn.Module):
    def __init__(self, fc_in_features=None):
        super(CustomResNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1)
        self.layer1 = BasicBlock(64, 64)
        self.pool0 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)

        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)
        self.layer2 = BasicBlock(64, 64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)

        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)
        self.layer3 = BasicBlock(64, 64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)

        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)
        self.layer4 = BasicBlock(64, 64)

        self.pool3 = nn.MaxPool2d(kernel_size=1, stride=2, padding=0)

        # Define a placeholder for the dynamically sized fully connected layer.
        self.fc = nn.Identity() if fc_in_features is None else nn.Linear(fc_in_features, 80 * 130)

    def forward(self, x):
        x = self.conv1(x)
        x = self.layer1(x)
        x = self.pool0(x)

        x = self.conv2(x)
        x = self.layer2(x)
        x = self.pool1(x)

        x = self.conv3(x)
        x = self.layer3(x)
        x = self.pool2(x)

        x = self.conv4(x)
        x = self.layer4(x)
        x = self.pool3(x)

        x = x.view(x.size(0), -1)
        x = self.fc(x)
        x = x.view(-1, 80, 130)
        return x

# Initialize the fully connected layer from the first input batch.
def handres_model(input_tensor):
    temp_model = CustomResNet()
    with torch.no_grad():
        x = temp_model.conv1(input_tensor)
        x = temp_model.layer1(x)
        x = temp_model.pool0(x)

        x = temp_model.conv2(x)
        x = temp_model.layer2(x)
        x = temp_model.pool1(x)

        x = temp_model.conv3(x)
        x = temp_model.layer3(x)
        x = temp_model.pool2(x)

        x = temp_model.conv4(x)
        x = temp_model.layer4(x)
        x = temp_model.pool3(x)

        flat_size = x.view(x.size(0), -1).size(1)

    return CustomResNet(fc_in_features=flat_size)

# Example usage.
input_tensor = torch.randn(1, 1, 8, 200)  # The channel dimension may vary, for example 18 or 70 channels.
model = handres_model(input_tensor)
output_tensor = model(input_tensor)
print(output_tensor.shape)  # Expected shape: [1, 80, 130].


In [ ]:
summary(
    model,
    input_size=(1, 1, 8, 200),
    col_names=["output_size", "num_params"])

In [ ]:

def collate_fn(batch):
    src_batch, tgt_batch = [], []
    for b in batch:
        src_batch.append(b['ECoG'])
        tgt_batch.append(b['num'])
    return src_batch, tgt_batch

def handres_model(channel):
    model = CustomResNet()
    # Replace the final fully connected layer.
    num_features = model.fc.in_features
    model.fc = nn.Linear(64 * int(channel) * 12, 80 * 130)
    return model

def normalize(array):
    """Normalizes the input array to the range [-1, 1]."""
    max_val = np.max(array)
    min_val = np.min(array)
    normalized_array = 2 * (array - min_val) / (max_val - min_val) - 1
    return normalized_array


In [ ]:
def load_data(num,covert_or_overt,elecs,ori_data):
    keys = [x for x in list(ori_data.keys()) if covert_or_overt in x and 'ma' not in x and 'na' not in x ]
    print(num,keys)
    num_all=len(keys)
    ECoG={}
    all_erps=[]
    labels=[]
    
    for i in range(num_all):
        if num in [45, 47 , 50 , 54]:
            ECoG[i]=ori_data[keys[i]][()][0:50,elecs,100:300]
        else:
            if num in [ 48 , 78] :
                #print(keys[i])
                ECoG[i]=ori_data[keys[i]][()][0:40,elecs,100:300]
            if num == 76 :
                ECoG[i]=ori_data[keys[i]][()][0:30,elecs,100:300]
                
            if num == 71 and covert_or_overt == 'ECoG_covert':
                ECoG[0]=ori_data['ECoG_covert_ba'][()][0:30,elecs,100:300]
                ECoG[1]=np.concatenate((ori_data['ECoG_covert_da'][()][0:15,elecs,100:300],
                                            ori_data['ECoG_covert_da'][()][0:15,elecs,100:300],),axis=0)
                ECoG[2]=ori_data['ECoG_covert_ga'][()][0:30,elecs,100:300]
                ECoG[3]=np.concatenate((ori_data['ECoG_covert_pa'][()][0:10,elecs,100:300],
                                            ori_data['ECoG_covert_pa'][()][10:20,elecs,100:300],
                                            ori_data['ECoG_covert_pa'][()][0:10,elecs,100:300]),axis=0)
                ECoG[4]=ori_data['ECoG_covert_ta'][()][0:30,elecs,100:300]
                ECoG[5]=ori_data['ECoG_covert_ka'][()][0:30,elecs,100:300]

                ECoG[6]=np.concatenate((ori_data['ECoG_covert_sa'][()][0:15,elecs,100:300],
                                            ori_data['ECoG_covert_sa'][()][0:15,elecs,100:300]),axis=0)

                ECoG[7]=np.concatenate((ori_data['ECoG_covert_sha'][()][0:10,elecs,100:300],
                                            ori_data['ECoG_covert_sha'][()][10:20,elecs,100:300],
                                            ori_data['ECoG_covert_sha'][()][0:10,elecs,100:300]),axis=0)
            if num == 71 and covert_or_overt == 'ECoG_overt':    
                ECoG[0]=ori_data['ECoG_overt_ba'][()][0:30,elecs,100:300]
                ECoG[1]=np.concatenate((ori_data['ECoG_overt_da'][()][0:15,elecs,100:300],
                                            ori_data['ECoG_overt_da'][()][0:15,elecs,100:300],),axis=0)
                ECoG[2]=ori_data['ECoG_overt_ga'][()][0:30,elecs,100:300]
                ECoG[3]=np.concatenate((ori_data['ECoG_overt_pa'][()][0:10,elecs,100:300],
                                            ori_data['ECoG_overt_pa'][()][10:20,elecs,100:300],
                                            ori_data['ECoG_overt_pa'][()][0:10,elecs,100:300]),axis=0)
                ECoG[4]=ori_data['ECoG_overt_ta'][()][0:30,elecs,100:300]
                ECoG[5]=ori_data['ECoG_overt_ka'][()][0:30,elecs,100:300]

                ECoG[6]=np.concatenate((ori_data['ECoG_overt_sa'][()][0:15,elecs,100:300],
                                            ori_data['ECoG_overt_sa'][()][0:15,elecs,100:300]),axis=0)

                ECoG[7]=np.concatenate((ori_data['ECoG_overt_sha'][()][0:10,elecs,100:300],
                                            ori_data['ECoG_overt_sha'][()][10:20,elecs,100:300],
                                            ori_data['ECoG_overt_sha'][()][0:10,elecs,100:300]),axis=0)
                
            if num == 73 and covert_or_overt == 'ECoG_covert':
                ECoG[0]=ori_data['ECoG_covert_ba'][()][0:30,elecs,100:300]
                ECoG[1]=ori_data['ECoG_covert_da'][()][0:30,elecs,100:300]
                ECoG[2]=ori_data['ECoG_covert_ga'][()][0:30,elecs,100:300]
                ECoG[3]=np.concatenate((ori_data['ECoG_covert_pa'][()][0:10,elecs,100:300],
                                            ori_data['ECoG_covert_pa'][()][0:10,elecs,100:300],
                                            ori_data['ECoG_covert_pa'][()][0:10,elecs,100:300]),axis=0)
                ECoG[4]=ori_data['ECoG_covert_ta'][()][0:30,elecs,100:300]
                ECoG[5]=np.concatenate((ori_data['ECoG_covert_ka'][()][0:20,elecs,100:300],
                                            ori_data['ECoG_covert_ka'][()][0:10,elecs,100:300]),axis=0)
                ECoG[6]=ori_data['ECoG_covert_sa'][()][0:30,elecs,100:300]
                ECoG[7]=ori_data['ECoG_covert_sha'][()][0:30,elecs,100:300]
            if num == 73 and covert_or_overt == 'ECoG_overt':
                ECoG[0]=ori_data['ECoG_overt_ba'][()][0:30,elecs,100:300]
                ECoG[1]=ori_data['ECoG_overt_da'][()][0:30,elecs,100:300]
                ECoG[2]=ori_data['ECoG_overt_ga'][()][0:30,elecs,100:300]
                ECoG[3]=np.concatenate((ori_data['ECoG_overt_pa'][()][0:10,elecs,100:300],
                                            ori_data['ECoG_overt_pa'][()][0:10,elecs,100:300],
                                            ori_data['ECoG_overt_pa'][()][0:10,elecs,100:300]),axis=0)
                ECoG[4]=ori_data['ECoG_overt_ta'][()][0:30,elecs,100:300]
                ECoG[5]=np.concatenate((ori_data['ECoG_overt_ka'][()][0:20,elecs,100:300],
                                            ori_data['ECoG_overt_ka'][()][0:10,elecs,100:300]),axis=0)
                ECoG[6]=ori_data['ECoG_overt_sa'][()][0:30,elecs,100:300]
                ECoG[7]=ori_data['ECoG_overt_sha'][()][0:30,elecs,100:300]
    input_shape=(len(elecs),ECoG[0].shape[2],1)
    for i in range(num_all):
        labels.append(i*np.ones(ECoG[i].shape[0]))
        all_erps.append(ECoG[i])
    all_erps = np.concatenate(all_erps, axis=0)
    labels = np.hstack(labels)
    print('HS=',num,all_erps.shape,len(labels))
    CV_sp=np.concatenate([ECoG[i] for i in range(num_all)],axis=0)
    hot=np.eye(num_all)
    all_hot=[]
    mix_TV={}
    for i in range(num_all):
        mix_TV[i]=np.tile((hot[i]),(ECoG[i].shape[0],1))
        #print(mix_TV[i].shape)
        all_hot.append(mix_TV[i])
    all_hot = np.concatenate(all_hot, axis=0)
    mix_sp=all_hot
    return CV_sp, mix_sp,num_all,input_shape

In [ ]:
def load_sound_data(num,covert_or_overt,ori_data):
    keys = [x for x in list(ori_data.keys()) if covert_or_overt in x and 'ma' not in x and 'na' not in x ]
    print(num,keys)
    num_all=len(keys)
    ECoG={}
    all_erps=[]
    labels=[]
    
    for i in range(num_all):
        if num in [45, 47 , 50 , 54]:
            ECoG[i]=ori_data[keys[i]][()][0:50]
        else:
            if num in [ 48 , 78] :
                #print(keys[i])
                ECoG[i]=ori_data[keys[i]][()][0:40]
            if num == 76 :
                ECoG[i]=ori_data[keys[i]][()][0:30]
                
            if num == 71 and covert_or_overt == 'soundClipMat_overt':
                ECoG[0]=ori_data['soundClipMat_overt_ba'][()][0:30]
                ECoG[1]=np.concatenate((ori_data['soundClipMat_overt_da'][()][0:15],
                                            ori_data['soundClipMat_overt_da'][()][0:15],),axis=0)
                ECoG[2]=ori_data['soundClipMat_overt_ga'][()][0:30]
                ECoG[3]=np.concatenate((ori_data['soundClipMat_overt_pa'][()][0:10],
                                            ori_data['soundClipMat_overt_pa'][()][10:20],
                                            ori_data['soundClipMat_overt_pa'][()][0:10]),axis=0)
                ECoG[4]=ori_data['soundClipMat_overt_ta'][()][0:30]
                ECoG[5]=ori_data['soundClipMat_overt_ka'][()][0:30]

                ECoG[6]=np.concatenate((ori_data['soundClipMat_overt_sa'][()][0:15],
                                            ori_data['soundClipMat_overt_sa'][()][0:15]),axis=0)

                ECoG[7]=np.concatenate((ori_data['soundClipMat_overt_sha'][()][0:10],
                                            ori_data['soundClipMat_overt_sha'][()][10:20],
                                            ori_data['soundClipMat_overt_sha'][()][0:10]),axis=0)
            if num == 71 and covert_or_overt == 'soundClipMat_overt':    
                ECoG[0]=ori_data['soundClipMat_overt_ba'][()][0:30]
                ECoG[1]=np.concatenate((ori_data['soundClipMat_overt_da'][()][0:15],
                                            ori_data['soundClipMat_overt_da'][()][0:15],),axis=0)
                ECoG[2]=ori_data['soundClipMat_overt_ga'][()][0:30]
                ECoG[3]=np.concatenate((ori_data['soundClipMat_overt_pa'][()][0:10],
                                            ori_data['soundClipMat_overt_pa'][()][10:20],
                                            ori_data['soundClipMat_overt_pa'][()][0:10]),axis=0)
                ECoG[4]=ori_data['soundClipMat_overt_ta'][()][0:30]
                ECoG[5]=ori_data['soundClipMat_overt_ka'][()][0:30]

                ECoG[6]=np.concatenate((ori_data['soundClipMat_overt_sa'][()][0:15],
                                            ori_data['soundClipMat_overt_sa'][()][0:15]),axis=0)

                ECoG[7]=np.concatenate((ori_data['soundClipMat_overt_sha'][()][0:10],
                                            ori_data['soundClipMat_overt_sha'][()][10:20],
                                            ori_data['soundClipMat_overt_sha'][()][0:10]),axis=0)
                
            if num == 73 and covert_or_overt == 'soundClipMat_overt':
                ECoG[0]=ori_data['soundClipMat_overt_ba'][()][0:30]
                ECoG[1]=ori_data['soundClipMat_overt_da'][()][0:30]
                ECoG[2]=ori_data['soundClipMat_overt_ga'][()][0:30]
                ECoG[3]=np.concatenate((ori_data['soundClipMat_overt_pa'][()][0:10],
                                            ori_data['soundClipMat_overt_pa'][()][0:10],
                                            ori_data['soundClipMat_overt_pa'][()][0:10]),axis=0)
                ECoG[4]=ori_data['soundClipMat_overt_ta'][()][0:30]
                ECoG[5]=np.concatenate((ori_data['soundClipMat_overt_ka'][()][0:20],
                                            ori_data['soundClipMat_overt_ka'][()][0:10]),axis=0)
                ECoG[6]=ori_data['soundClipMat_overt_sa'][()][0:30]
                ECoG[7]=ori_data['soundClipMat_overt_sha'][()][0:30]
            if num == 73 and covert_or_overt == 'soundClipMat_overt':
                ECoG[0]=ori_data['soundClipMat_overt_ba'][()][0:30]
                ECoG[1]=ori_data['soundClipMat_overt_da'][()][0:30]
                ECoG[2]=ori_data['soundClipMat_overt_ga'][()][0:30]
                ECoG[3]=np.concatenate((ori_data['soundClipMat_overt_pa'][()][0:10],
                                            ori_data['soundClipMat_overt_pa'][()][0:10],
                                            ori_data['soundClipMat_overt_pa'][()][0:10]),axis=0)
                ECoG[4]=ori_data['soundClipMat_overt_ta'][()][0:30]
                ECoG[5]=np.concatenate((ori_data['soundClipMat_overt_ka'][()][0:20],
                                            ori_data['soundClipMat_overt_ka'][()][0:10]),axis=0)
                ECoG[6]=ori_data['soundClipMat_overt_sa'][()][0:30]
                ECoG[7]=ori_data['soundClipMat_overt_sha'][()][0:30]
                
    for i in range(num_all):
        labels.append(i*np.ones(ECoG[i].shape[0]))
        all_erps.append(ECoG[i])
    all_erps = np.concatenate(all_erps, axis=0)
    labels = np.hstack(labels)
    #print('HS=',num,all_erps.shape,len(labels))
    CV_sp=np.concatenate([ECoG[i] for i in range(num_all)],axis=0)
    
    


    
    
    n_mels=150
    soundSampleFreq=24414
    fmax=10000
    fmin=20
    n_fft=2270
    hop_length=int(n_fft/4)
    mel_group=[]
    for i in CV_sp:
        temp_mel_temp = librosa.resample(i[int(np.round(97656*1/4)):int(np.round(97656*5/8))]
                                                       ,   orig_sr=soundSampleFreq, target_sr=22050)
        temp_mel_temp = normalize(temp_mel_temp)
        wav = torch.tensor(temp_mel_temp, dtype=torch.float32).unsqueeze(0)
        mel = stft.mel_spectrogram(wav)
        mel_group.append(np.squeeze(mel.numpy()))
    
    mel_all=np.array(mel_group)
    
    print('HS=',num,mel_all.shape,len(labels))

    return mel_all

In [ ]:
import numpy as np

def pad_to_8(arr):
    """
    Ensure input array has shape (a, b, 200) and pad to (a, 8, 200) if b < 8.
    """
    a, b, c = arr.shape
    if b >= 8:
        return arr[:, :, :]  # Truncate the sequence when b exceeds 8.
    else:
        pad_width = ((0, 0), (0, 8 - b), (0, 0))  # Pad the second dimension.
        return np.pad(arr, pad_width, mode='constant', constant_values=0)


In [ ]:
import json
import os
from pathlib import Path


def _find_repository_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "model_code" / "classifier_and_speech_synthesizer" / "paths.py").is_file():
            return candidate
    raise FileNotFoundError("Cannot locate the repository root")


repository_root = _find_repository_root()
workspace_root = Path(
    os.environ.get("COVERT_READING_WORKSPACE", repository_root / "workspace")
).expanduser().resolve()
data_root = Path(
    os.environ.get("COVERT_READING_DATA_ROOT", workspace_root / "model_data")
).expanduser().resolve()
electrode_list_path = Path(
    os.environ.get(
        "COVERT_READING_ELECTRODE_LIST",
        workspace_root / "private" / "electrode_lists.json",
    )
).expanduser().resolve()
checkpoint_dir = Path(
    os.environ.get("COVERT_READING_SPEECH_CHECKPOINT_DIR", workspace_root / "speech_checkpoints")
).expanduser().resolve()
result_dir = workspace_root / "model_results" / "speech"

if not electrode_list_path.is_file():
    raise FileNotFoundError(f"Electrode list file not found: {electrode_list_path}")
with electrode_list_path.open(encoding="utf-8") as handle:
    electrode_lists = json.load(handle)

checkpoint_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)

start_time=time.time()
MCD_all={}
HS=[45, 47,  48,  50,  54, 71, 73,  76, 78]
for band in ['hg','b1']:
    MCD_all[band]={}
    
    for o_or_v in ['SA','SI']:
        MCD_all[band][o_or_v]={}
        classifier_electrodes = electrode_lists["classifier"]
        if o_or_v=='SA':
            sound_covert_or_overt='soundClipMat_overt'
            covert_or_overt='ECoG_overt'
        elif o_or_v == 'SI':
            sound_covert_or_overt='soundClipMat_overt'
            covert_or_overt='ECoG_covert'
        
        for num in HS:
            MCD_all[band][o_or_v][num]=[]
            elecs=classifier_electrodes[o_or_v][band][str(num)]
            if band =='b1':
                ori_data=scio.loadmat(data_root / "HSblockdata" / f"HS{num}_Block_overt_covert_12_24_zscore_100Hz.mat")
            elif band == 'hg':
                ori_data=scio.loadmat(data_root / "HSblockdata" / f"HS{num}_Block_overt_covert_70_150_zscore_100Hz.mat")
                
            CV_sp,mix_sp,num_all,input_shape=load_data(num,covert_or_overt,elecs,ori_data)
            CV_sp=pad_to_8(CV_sp)
            sound=load_sound_data(num,sound_covert_or_overt,ori_data)
            
            
            print(band,o_or_v,num)
            kf = KFold(n_splits=10, shuffle=True, random_state=42)
            for train_idx, test_idx in kf.split(CV_sp):
                ECoG_train, ECoG_test=CV_sp[train_idx],CV_sp[test_idx]
                sound_train, sound_test=sound[train_idx],sound[test_idx]
                train_set = pd.DataFrame({
                    'ECoG': list(ECoG_train),
                    'num': list(sound_train)
                })
                
                test_set = pd.DataFrame({
                    'ECoG': list(ECoG_test),
                    'num': list(sound_test)
                })
                trainData=dsds.from_dict(train_set)
                testData=dsds.from_dict(test_set)
            

            
            
                collate_fn = partial(collate_fn)
                dataloader = DataLoader(trainData, batch_size=1,  collate_fn=collate_fn, shuffle=True)

                model = handres_model(torch.randn(1,1,CV_sp.shape[1],200))
                model.train()
                optimizer = optim.NAdam(model.parameters(),lr=0.0001,betas=(0.9, 0.999),eps=1e-08,
                                    weight_decay=0.002)
                mean_loss_mean=[]
                start_time = time.time()
                for i in tqdm(range(120)):
                    mean_loss=[]
                    
                    saveloss=[]
                    for a,b in dataloader:
                        optimizer.zero_grad()
                        src=np.array(a)
                        src=np.expand_dims(src,axis=1)
                    #src=src.reshape(src.shape[0],1,elecs,200)
                        inputs=torch.tensor(src, dtype=torch.float32).to(device)
                        #label=np.squeeze(np.array(b))
                        labels=torch.tensor(b, dtype=torch.float32).to(device)
                        model=model.to(device)
                        outputs= model(inputs)
                        #mintr=outputs.min()
                        criterion = nn.MSELoss().to(device)
                        loss = criterion(outputs, labels)
                        loss.backward()
                        optimizer.step()

                        #print(loss.item())
                    #print(f'Epoch {i}, Loss: {loss.item()}, Accuracy: {accuracy.item()}')

                    mean_loss.append(loss.item())
                    mean_loss_mean.append(np.mean(mean_loss))
                    if i >1 and mean_loss<np.min(mean_loss_mean[:-1]):
                        saveloss=mean_loss_mean[i]
                        torch.save({'epoch': i,'model_state_dict': model.state_dict(),
                                    'optimizer_state_dict': optimizer.state_dict(),'loss': loss,}, 
                                    str(checkpoint_dir / f'{band}_{o_or_v}_{num}checkpoint_best.pth'))
                    else:
                        if i ==1 :
                            print(mean_loss_mean)
                            saveloss=mean_loss_mean[1]

                    
                    print(f'Epoch {i},meanLoss:{np.mean(mean_loss)}','saveloss:',saveloss)
                    torch.save({'epoch': i,'model_state_dict': model.state_dict(),
                                    'optimizer_state_dict': optimizer.state_dict(),'loss': loss,}, 
                                    str(checkpoint_dir / f'{band}_{o_or_v}_{num}checkpoint_last.pth'))
                    
                # model_best = model
                # checkpoint = torch.load(str(checkpoint_dir / f'{band}_{o_or_v}_{num}checkpoint_last.pth'))
                # epoch = checkpoint['epoch']
                # print(epoch)
                # model_best.load_state_dict(checkpoint['model_state_dict'])
                # model_best.eval()
                for i in range(len(test_set["ECoG"])):
                    testECoG_new=test_set["ECoG"][i].reshape(1,1,test_set["ECoG"][0].shape[0],200)
                    testECoG_new.shape
                    testECoG_new_tensor=torch.tensor(testECoG_new, dtype=torch.float32).to(device)
                    testECoG_new_tensor.shape
                    out_feats_GPU= model(testECoG_new_tensor)
                    temel=out_feats_GPU.detach().cpu().numpy() 
                    temel=np.squeeze(temel)
                    MCD_temp=get_metrics_mels(temel,test_set["num"][i], n_mfcc= 1, take_log = False, use_dtw= True)
                    MCD_all[band][o_or_v][num].append(MCD_temp[0])
                print(band, o_or_v, num, MCD_all[band][o_or_v][num])
                break

np.save(result_dir / 'all_MCD_final.npy', MCD_all, allow_pickle=True)
end_time = time.time()     
runtime=end_time - start_time
print( "training execution time:" +'%.3f'% (runtime) + "seconds")